# 2. Definition of Default

| | |
|---|---|
| **Aim** | Determine the date and reason for the first default for each loan |
| **Input data** | `clean/` from notebook 1 |
| **Result** | `default/` with default columns |
| **Basis** | Article 178 of the CRR, Delegated Regulation (EU) 2018/171, EBA/GL/2020/02 |

## Definition

A loan is in default in the month in which the **first** of these events occurs:

| Reason | Condition |
|---|---|
| **DPD** | Deliquency exceed EUR 100 **and** 1% of the balance for 3 consecutive months, and the delinquency status is at least 3 (90 days) |
| **Sale at a loss** | The loan ended with a loss sale (codes 02, 03, 09, 15) |
| **Modification** | Loan terms have been changed |

## How do we calculate default DPD?

1. **Balance and installment.** Annuity installment based on current balance; the hidden balance in the first few months from the schedule.
2. **Arrears.** The data does not provide the amount of arrears, so I assume: number of arrears × installment.
3. **Threshold.** Check whether the arrears exceed both thresholds simultaneously.
4. **Number.** Count of the subsequent months above the thresholds. When the arrears fall below the threshold, the counter returns to zero. In COVID forbearance, the counter remains stationary.
5. **First default.** The earliest event.

The calculations are in part B of the `functions.py` file.

In [1]:
import polars as pl

import config as C
import functions as F

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(70)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_hide_column_data_types(True)
C.REPORT_DIR.mkdir(parents=True, exist_ok=True)

## Examples: How the Definition Works

### Example 1. Typical 30-Year Loan

One past-due payment is below the 1% threshold, so the counter starts at status 2, and default occurs at status 4.

In [2]:
F.example_table(F.example_loan([0, 1, 2, 3, 4, 5]))

month,status,payment_usd,overdue_eur,share_pct,above_threshold,counter,default,reason
"""2019-01""",0,1289.0,0.0,0.0,false,0,false,null
"""2019-02""",1,1290.0,1173.0,0.65,false,0,false,null
"""2019-03""",2,1292.0,2350.0,1.29,true,1,false,null
"""2019-04""",3,1294.0,3530.0,1.94,true,2,false,null
"""2019-05""",4,1296.0,4713.0,2.59,true,3,true,"""DPD"""
"""2019-06""",5,1298.0,5900.0,3.25,true,4,false,null


### Example 2. Repayment resets the counter

A loan with a high monthly payment exceeds the thresholds from the first overdue payment. Repayment in the third month resets the counter.

In [3]:
F.example_table(F.example_loan([1, 2, 0, 1, 2, 3], balance=20_000, rate=0, remaining=12))

month,status,payment_usd,overdue_eur,share_pct,above_threshold,counter,default,reason
"""2019-01""",1,1667.0,1515.0,8.33,true,1,false,null
"""2019-02""",2,1818.0,3306.0,18.18,true,2,false,null
"""2019-03""",0,2000.0,0.0,0.0,false,0,false,null
"""2019-04""",1,2222.0,2020.0,11.11,true,1,false,null
"""2019-05""",2,2500.0,4545.0,25.0,true,2,false,null
"""2019-06""",3,2857.0,7792.0,42.86,true,3,true,"""DPD"""


### Example 3. Status remains at 2

The borrower pays one monthly installment but doesn't catch up. The counter increases, but the oldest installment is not yet 90 days old, so default only occurs at status 3.

In [4]:
F.example_table(F.example_loan([2, 2, 2, 2, 3], balance=20_000, rate=0, remaining=12))

month,status,payment_usd,overdue_eur,share_pct,above_threshold,counter,default,reason
"""2019-01""",2,1667.0,3030.0,16.67,true,1,false,null
"""2019-02""",2,1818.0,3306.0,18.18,true,2,false,null
"""2019-03""",2,2000.0,3636.0,20.0,true,3,false,null
"""2019-04""",2,2222.0,4040.0,22.22,true,4,false,null
"""2019-05""",3,2500.0,6818.0,37.5,true,5,true,"""DPD"""


### Example 4: COVID Forbearance

From April to September 2020, the status increases, but the counter remains stagnant. After the forbearance ends, the arrears continue to exceed the thresholds, so the counter starts and defaults.

In [5]:
import datetime as dt

covid = [None, "F", "F", "F", "F", "F", "F", None, None, None]
F.example_table(F.example_loan(list(range(10)), balance=20_000, rate=0, remaining=120,
                               start=dt.date(2020, 3, 1), forbearance=covid))

month,status,payment_usd,overdue_eur,share_pct,above_threshold,counter,default,reason
"""2020-03""",0,167.0,0.0,0.0,false,0,false,null
"""2020-04""",1,168.0,153.0,0.84,false,0,false,null
"""2020-05""",2,169.0,308.0,1.69,true,0,false,null
"""2020-06""",3,171.0,466.0,2.56,true,0,false,null
"""2020-07""",4,172.0,627.0,3.45,true,0,false,null
"""2020-08""",5,174.0,791.0,4.35,true,0,false,null
"""2020-09""",6,175.0,957.0,5.26,true,0,false,null
"""2020-10""",7,177.0,1126.0,6.19,true,1,false,null
"""2020-11""",8,179.0,1299.0,7.14,true,2,false,null
"""2020-12""",9,180.0,1474.0,8.11,true,3,true,"""DPD"""


## Running on data

In [6]:
C.DEFAULT_DIR.mkdir(parents=True, exist_ok=True)
summary = []

for path in sorted(C.CLEAN_DIR.glob("*.parquet")):
    df = F.build_default(pl.read_parquet(path))
    df.write_parquet(C.DEFAULT_DIR / path.name)
    loans = df["LOAN_ID"].n_unique()
    defaults = int(df["default_flag"].sum())
    summary.append({"cohort": path.stem, "loans": loans, "defaults": defaults,
                    "share_pct": round(defaults / loans * 100, 2)})
pl.DataFrame(summary)

cohort,loans,defaults,share_pct
"""2007Q1""",6332,1142,18.04
"""2019Q1""",8721,257,2.95


## Share - default causes

In [7]:
panel = pl.scan_parquet(C.DEFAULT_DIR / "*.parquet")
reasons = (panel.filter(pl.col("default_flag"))
           .group_by("cohort", "default_reason").len()
           .sort("cohort", "len", descending=[False, True])
           .rename({"cohort": "cohort", "default_reason": "reason", "len": "loans"}).collect())
reasons.write_csv(C.REPORT_DIR / "defaults_by_reason.csv")
reasons

cohort,reason,loans
"""2007Q1""","""DPD""",1091
"""2007Q1""","""UTP_modification""",30
"""2007Q1""","""UTP_loss_sale""",21
"""2019Q1""","""DPD""",255
"""2019Q1""","""UTP_modification""",2
